Dataset Download

In [ ]:
# ============================================================
# GOOGLE DRIVE AUTHENTICATION
# Run this at the START of the notebook
# ============================================================

!pip install -q PyDrive

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate user
auth.authenticate_user()

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()

drive = GoogleDrive(gauth)

print("✅ Google Drive authentication successful")

In [ ]:
!pip install opencv-python matplotlib lxml
!wget https://zenodo.org/api/records/5106795/files-archive
!unzip -o files-archive
!unzip -o validation_set
!unzip -o train_set
!unzip -o test_set
!rm /kaggle/working/*.zip
!rm /kaggle/working/files-archive

Dataset -> Yolo Dataset

In [ ]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

# Root from your screenshot
data_root = Path("/kaggle/working")

CLASS_NAMES = [
    "CYPRO","CYPRO_max","CYPRO_min","ECHCG","ECHCG_V1","ECHCG_V2",
    "ECHCG_Ve","NC","OE","POROL","SOLNI","SOLNI_V1","SOLNI_V2",
    "SOLNI_Vc","ZEAMX","ZEAMX_V1","ZEAMX_V3","ZEAMX_V4"
]

def count_xml_labels(folder_path):
    counts = Counter()
    # Find all XML files in the folder (including subfolders just in case)
    xml_files = list(folder_path.rglob("*.xml"))

    for xml_file in xml_files:
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()

            # Find every <object> tag in the XML
            for obj in root.findall('object'):
                name = obj.find('name').text
                if name in CLASS_NAMES:
                    counts[name] += 1
                else:
                    # In case there's a typo in the XML names vs your list
                    counts[name] += 1
        except Exception as e:
            continue # Skip corrupted files
    return counts

# Mapping based on your screenshot
splits = {
    "Train": data_root / "train_set",
    "Val":   data_root / "validation_set",
    "Test":  data_root / "test_set"
}

# Run the counter
results = {split_name: count_xml_labels(path) for split_name, path in splits.items()}

# Print Results
header = f"{'Class':<15} {'Train':>8} {'Val':>8} {'Test':>8}"
print(header)
print("-" * len(header))

for name in CLASS_NAMES:
    t = results["Train"].get(name, 0)
    v = results["Val"].get(name, 0)
    te = results["Test"].get(name, 0)
    print(f"{name:<15} {t:>8} {v:>8} {te:>8}")

STRATIFYING LOGIC - DO NOT TOUCH

In [ ]:
"""
WeedMaize — Robust Multi-Label Stratified Dataset Splitter
===========================================================
Problem solved:
  - Dominant-label stratification ignores minority classes inside images
  - Pre-split folders carry historical bias
  - Rare classes randomly fall into only train or only test

Solution:
  - Iterative multi-label stratification (Sechidis et al., 2011)
    via skmultilearn — every class in every image is balanced
  - Fallback: guaranteed seeding of rare classes into ALL splits
    before stratification runs
  - Class weights computed and exported for use in your loss function
"""

import os
import shutil
import json
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

# ── Optional imports (graceful fallback) ────────────────────────────────────
try:
    from skmultilearn.model_selection import IterativeStratification
    HAS_SKMULTILEARN = True
except ImportError:
    HAS_SKMULTILEARN = False
    print("⚠️  skmultilearn not found — using fallback multi-label stratifier.")
    print("    Install with: pip install scikit-multilearn")

# ═══════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

ROOT       = Path("/kaggle/working")          # train_set / val_set / test_set live here
NEW_ROOT   = Path("/kaggle/working/stratified_dataset")  # output goes here

# Split ratios — must sum to 1.0
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

RANDOM_SEED = 42

# Minimum samples a class must have in EACH split after stratification.
# Images containing only ultra-rare classes are force-seeded first.
MIN_PER_SPLIT = 5

# ── Label merge map ─────────────────────────────────────────────────────────
# Classes absent from this map are DROPPED silently.
LABEL_MAP = {
    "CYPRO":     "CYPRO",
    "CYPRO_max": "CYPRO_max",
    "CYPRO_min": "CYPRO_min",

    "ECHCG":    "ECHCG",
    "ECHCG_V1": "ECHCG_Ve",   # merge V1 + V2 into Ve
    "ECHCG_V2": "ECHCG_Ve",
    "ECHCG_Ve": "ECHCG_Ve",

    "NC": "NC",

    "SOLNI":    "SOLNI_V1",   # merge base + V1 + V2
    "SOLNI_V1": "SOLNI_V1",
    "SOLNI_V2": "SOLNI_V1",
    "SOLNI_Vc": "SOLNI_Vc",   # keep separate — very numerous

    "ZEAMX":    "ZEAMX_V1",   # merge base into V1
    "ZEAMX_V1": "ZEAMX_V1",
    "ZEAMX_V3": "ZEAMX_V3",
    "ZEAMX_V4": "ZEAMX_V4",

    # OE, POROL — intentionally excluded
}

FINAL_CLASSES = sorted(set(LABEL_MAP.values()))
CLS_TO_IDX    = {c: i for i, c in enumerate(FINAL_CLASSES)}
N_CLASSES     = len(FINAL_CLASSES)


# ═══════════════════════════════════════════════════════════════════════════
# 2. COLLECT ALL IMAGE-ANNOTATION PAIRS
# ═══════════════════════════════════════════════════════════════════════════

def collect_all_pairs() -> list[dict]:
    """Scan all three original folders, parse XML, map labels."""
    all_data, skipped = [], 0
    print("\n── Scanning original dataset folders ──────────────────────────────")

    for folder in ["train_set", "validation_set", "test_set"]:
        folder_path = ROOT / folder
        if not folder_path.exists():
            print(f"  ⚠️  Not found, skipping: {folder_path}")
            continue

        xmls = sorted(folder_path.glob("*.xml"))
        print(f"  {folder}: {len(xmls)} XML files found")

        for xml_file in xmls:
            img_file = xml_file.with_suffix(".jpg")
            if not img_file.exists():
                img_file = xml_file.with_suffix(".JPG")
            if not img_file.exists():
                skipped += 1
                continue

            try:
                tree    = ET.parse(xml_file)
                root_el = tree.getroot()

                # All mapped labels in this image (one per bounding box)
                mapped = [
                    LABEL_MAP[obj.find("name").text]
                    for obj in root_el.findall("object")
                    if obj.find("name") is not None
                    and obj.find("name").text in LABEL_MAP
                ]
                if not mapped:
                    skipped += 1
                    continue

                # Multi-hot vector — presence of each class in this image
                multi_hot = np.zeros(N_CLASSES, dtype=np.int8)
                for lbl in mapped:
                    multi_hot[CLS_TO_IDX[lbl]] = 1

                # Box-level counts per class (used for class weights later)
                box_counts = Counter(mapped)

                all_data.append({
                    "xml":       xml_file,
                    "img":       img_file,
                    "stem":      f"{folder}__{xml_file.stem}",
                    "multi_hot": multi_hot,          # shape (N_CLASSES,)
                    "box_counts": box_counts,
                })

            except Exception as e:
                print(f"  ⚠️  Parse error {xml_file.name}: {e}")
                skipped += 1

    print(f"\n✅  Collected {len(all_data)} valid pairs. Skipped {skipped}.")
    return all_data


# ═══════════════════════════════════════════════════════════════════════════
# 3. PRINT CLASS DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════════════

def print_distribution(all_data: list[dict], title: str = "Full dataset"):
    """Show image-level and box-level counts per class."""
    img_counts = Counter()
    box_counts  = Counter()
    for d in all_data:
        for cls, idx in CLS_TO_IDX.items():
            if d["multi_hot"][idx]:
                img_counts[cls] += 1
        for cls, cnt in d["box_counts"].items():
            box_counts[cls] += cnt

    total_imgs = len(all_data)
    total_boxes = sum(box_counts.values())

    print(f"\n{'─'*70}")
    print(f"  {title}  ({total_imgs} images | {total_boxes} bounding boxes)")
    print(f"{'─'*70}")
    print(f"  {'Class':<15} {'Images':>8} {'Img%':>7}  {'Boxes':>8} {'Box%':>7}")
    print(f"  {'─'*15} {'─'*8} {'─'*7}  {'─'*8} {'─'*7}")
    for cls in FINAL_CLASSES:
        ic = img_counts.get(cls, 0)
        bc = box_counts.get(cls, 0)
        flag = "  ⚠️ RARE" if ic < MIN_PER_SPLIT * 3 else ""
        print(f"  {cls:<15} {ic:>8} {ic/total_imgs*100:>6.1f}%  "
              f"{bc:>8} {bc/total_boxes*100:>6.1f}%{flag}")
    print(f"{'─'*70}")
    return img_counts, box_counts


# ═══════════════════════════════════════════════════════════════════════════
# 4. RARE-CLASS FORCE SEEDING
# ═══════════════════════════════════════════════════════════════════════════

def force_seed_rare_classes(
    all_data: list[dict],
    img_counts: Counter,
) -> tuple[list[int], list[int], list[int], list[int]]:
    """
    For any class with fewer than MIN_PER_SPLIT * 3 images, guarantee
    at least MIN_PER_SPLIT samples land in each split by reserving them
    before stratification runs.

    Returns: (train_seed, val_seed, test_seed, remaining_indices)
    """
    rare_classes = [
        cls for cls in FINAL_CLASSES
        if img_counts.get(cls, 0) < MIN_PER_SPLIT * 3
    ]

    if not rare_classes:
        print("\n✅  No rare classes detected — skipping force-seed step.")
        return [], [], [], list(range(len(all_data)))

    print(f"\n⚠️  Rare classes detected (< {MIN_PER_SPLIT * 3} images): {rare_classes}")
    print("    Force-seeding these across all splits before stratification...")

    train_seed, val_seed, test_seed = [], [], []
    seeded = set()

    for cls in rare_classes:
        cls_idx = CLS_TO_IDX[cls]
        # Indices of images that contain this rare class
        candidates = [
            i for i, d in enumerate(all_data)
            if d["multi_hot"][cls_idx] and i not in seeded
        ]
        np.random.shuffle(candidates)  # reproducible via seed set before call

        n = len(candidates)
        # Proportional seeding respecting ratios
        n_train = max(MIN_PER_SPLIT, round(n * TRAIN_RATIO))
        n_val   = max(1, round(n * VAL_RATIO))
        n_test  = max(1, n - n_train - n_val)
        # Clip if not enough
        n_train = min(n_train, n - 2)
        n_val   = min(n_val,   n - n_train - 1)
        n_test  = n - n_train - n_val

        for i in candidates[:n_train]:
            train_seed.append(i); seeded.add(i)
        for i in candidates[n_train:n_train + n_val]:
            val_seed.append(i);   seeded.add(i)
        for i in candidates[n_train + n_val:n_train + n_val + n_test]:
            test_seed.append(i);  seeded.add(i)

        actual_seeded = candidates[:n_train + n_val + n_test]
        print(f"    {cls}: seeded {n_train} train | {n_val} val | {n_test} test "
              f"(of {n} total)")

    remaining = [i for i in range(len(all_data)) if i not in seeded]
    print(f"\n  Seeded {len(seeded)} images. {len(remaining)} remain for stratification.")
    return train_seed, val_seed, test_seed, remaining


# ═══════════════════════════════════════════════════════════════════════════
# 5A. ITERATIVE MULTI-LABEL STRATIFICATION (preferred)
# ═══════════════════════════════════════════════════════════════════════════

def iterative_stratify(
    indices: list[int],
    all_data: list[dict],
) -> tuple[list[int], list[int], list[int]]:
    """Use skmultilearn IterativeStratification on remaining pool."""
    X = np.array(indices).reshape(-1, 1)
    Y = np.vstack([all_data[i]["multi_hot"] for i in indices])

    # Two-step: split off test, then split remainder into train/val
    test_size  = TEST_RATIO / (TRAIN_RATIO + VAL_RATIO + TEST_RATIO)
    val_size_2 = VAL_RATIO  / (TRAIN_RATIO + VAL_RATIO)

    # Pre-shuffle for reproducibility — IterativeStratification does not
    # accept random_state when shuffle=False (its only supported mode)
    rng = np.random.default_rng(RANDOM_SEED)
    perm = rng.permutation(len(indices))
    X, Y = X[perm], Y[perm]

    strat1 = IterativeStratification(
        n_splits=2,
        order=2,
        sample_distribution_per_fold=[test_size, 1 - test_size],
    )
    for train_val_pos, test_pos in strat1.split(X, Y):
        break  # only need first fold

    X_tv, Y_tv = X[train_val_pos], Y[train_val_pos]

    strat2 = IterativeStratification(
        n_splits=2,
        order=2,
        sample_distribution_per_fold=[val_size_2, 1 - val_size_2],
    )
    for train_pos, val_pos in strat2.split(X_tv, Y_tv):
        break

    train_idx = [X_tv[i, 0] for i in train_pos]
    val_idx   = [X_tv[i, 0] for i in val_pos]
    test_idx  = [X[i, 0] for i in test_pos]

    return train_idx, val_idx, test_idx


# ═══════════════════════════════════════════════════════════════════════════
# 5B. FALLBACK: GREEDY MULTI-LABEL STRATIFICATION
# ═══════════════════════════════════════════════════════════════════════════

def greedy_multilabel_stratify(
    indices: list[int],
    all_data: list[dict],
) -> tuple[list[int], list[int], list[int]]:
    """
    Greedy fallback when skmultilearn is unavailable.
    Assigns each image to the split where its rarest class is most under-represented.
    Substantially better than dominant-label single-label stratification.
    """
    rng = np.random.default_rng(RANDOM_SEED)
    idx_arr = np.array(indices)
    rng.shuffle(idx_arr)

    # Track how many images of each class are in each split
    cls_counts = {
        "train": np.zeros(N_CLASSES, dtype=int),
        "val":   np.zeros(N_CLASSES, dtype=int),
        "test":  np.zeros(N_CLASSES, dtype=int),
    }
    split_total = {"train": 0, "val": 0, "test": 0}
    split_map   = {"train": [], "val": [], "test": []}

    target = {
        "train": TRAIN_RATIO,
        "val":   VAL_RATIO,
        "test":  TEST_RATIO,
    }
    n_total = len(idx_arr)

    for i in idx_arr:
        hot    = all_data[i]["multi_hot"]
        active = np.where(hot)[0]   # classes present in this image

        best_split, best_score = None, float("inf")
        for split_name in ["train", "val", "test"]:
            # Penalise if this split is already over its target ratio
            size_penalty = split_total[split_name] / max(n_total * target[split_name], 1)

            # For each active class, compute how under-represented it is here
            deficit = 0.0
            for ci in active:
                total_of_cls = sum(cls_counts[s][ci] for s in ["train", "val", "test"])
                if total_of_cls == 0:
                    deficit += 1.0   # unseen class — high priority
                else:
                    expected = target[split_name] * total_of_cls
                    actual   = cls_counts[split_name][ci]
                    deficit += max(0.0, expected - actual) / total_of_cls

            # Lower score = better (more deficit, less size penalty)
            score = size_penalty - deficit / max(len(active), 1)

            if score < best_score:
                best_score, best_split = score, split_name

        split_map[best_split].append(i)
        split_total[best_split] += 1
        for ci in active:
            cls_counts[best_split][ci] += 1

    return split_map["train"], split_map["val"], split_map["test"]


# ═══════════════════════════════════════════════════════════════════════════
# 6. COMPUTE CLASS WEIGHTS (for loss balancing during training)
# ═══════════════════════════════════════════════════════════════════════════

def compute_class_weights(train_indices: list[int], all_data: list[dict]) -> dict:
    """
    Inverse-frequency weights on BOUNDING BOX counts in the training set.
    Export as JSON for direct use in your training script.
    """
    box_counts = Counter()
    for i in train_indices:
        for cls, cnt in all_data[i]["box_counts"].items():
            box_counts[cls] += cnt

    total = sum(box_counts.values())
    n_cls = len(FINAL_CLASSES)

    # Balanced inverse-frequency: w_i = total / (n_classes * count_i)
    weights = {}
    for cls in FINAL_CLASSES:
        cnt = box_counts.get(cls, 0)
        if cnt == 0:
            weights[cls] = 0.0   # shouldn't happen after seeding
        else:
            weights[cls] = round(total / (n_cls * cnt), 4)

    return weights


# ═══════════════════════════════════════════════════════════════════════════
# 7. COPY / SYMLINK FILES
# ═══════════════════════════════════════════════════════════════════════════

def create_split_files(
    indices: list[int],
    all_data: list[dict],
    split_name: str,
    use_symlinks: bool = True,
):
    """Write image + XML into split folder (symlink or hard copy)."""
    dest = NEW_ROOT / split_name
    dest.mkdir(parents=True, exist_ok=True)

    for i in indices:
        item = all_data[i]
        xml_dst = dest / f"{item['stem']}.xml"
        img_dst = dest / f"{item['stem']}{item['img'].suffix}"

        for src, dst in [(item["xml"], xml_dst), (item["img"], img_dst)]:
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            if use_symlinks:
                os.symlink(src.resolve(), dst)
            else:
                shutil.copy2(src, dst)

    print(f"  ✅ {split_name:<6}: {len(indices)} images → {dest}")


# ═══════════════════════════════════════════════════════════════════════════
# 8. SPLIT VALIDATION — catch mismatches BEFORE training
# ═══════════════════════════════════════════════════════════════════════════

def validate_splits(
    train_idx: list[int],
    val_idx:   list[int],
    test_idx:  list[int],
    all_data:  list[dict],
) -> bool:
    """
    Assert:
      1. No image appears in more than one split
      2. Every class present in test/val also exists in train
      3. No split has 0 images of any class that exists in the full dataset
    """
    print("\n── Validating splits ───────────────────────────────────────────────")
    ok = True

    # (1) Overlap check
    sets = [set(train_idx), set(val_idx), set(test_idx)]
    for a, b, name in [(sets[0], sets[1], "train∩val"),
                       (sets[0], sets[2], "train∩test"),
                       (sets[1], sets[2], "val∩test")]:
        overlap = a & b
        if overlap:
            print(f"  ❌ OVERLAP in {name}: {len(overlap)} images shared!")
            ok = False
        else:
            print(f"  ✅ No overlap: {name}")

    # (2 & 3) Class coverage
    def cls_set(idxs):
        s = set()
        for i in idxs:
            s.update(c for c, v in zip(FINAL_CLASSES, all_data[i]["multi_hot"]) if v)
        return s

    train_cls = cls_set(train_idx)
    val_cls   = cls_set(val_idx)
    test_cls  = cls_set(test_idx)
    all_cls   = cls_set(list(range(len(all_data))))

    missing_val  = all_cls - val_cls
    missing_test = all_cls - test_cls
    missing_train_but_in_test = (test_cls | val_cls) - train_cls

    if missing_val:
        print(f"  ⚠️  Classes missing from VAL:  {missing_val}")
    else:
        print(f"  ✅ All classes present in val")

    if missing_test:
        print(f"  ⚠️  Classes missing from TEST: {missing_test}")
    else:
        print(f"  ✅ All classes present in test")

    if missing_train_but_in_test:
        print(f"  ❌ CRITICAL: Classes in test/val but NOT in train: "
              f"{missing_train_but_in_test}")
        ok = False
    else:
        print(f"  ✅ Train contains all classes seen in val/test")

    print(f"{'─'*70}")
    return ok


# ═══════════════════════════════════════════════════════════════════════════
# 9. MAIN ORCHESTRATION
# ═══════════════════════════════════════════════════════════════════════════

def main():
    np.random.seed(RANDOM_SEED)

    # ── Step 1: Collect data ──────────────────────────────────────────────
    all_data = collect_all_pairs()
    if not all_data:
        raise RuntimeError("No data found — check ROOT path.")

    # ── Step 2: Show full distribution ───────────────────────────────────
    img_counts, box_counts = print_distribution(all_data, "Full merged dataset")

    # ── Step 3: Force-seed rare classes ──────────────────────────────────
    train_seed, val_seed, test_seed, remaining = force_seed_rare_classes(
        all_data, img_counts
    )

    # ── Step 4: Stratify remaining pool ──────────────────────────────────
    if len(remaining) == 0:
        # Edge case: entire dataset was rare-seeded
        train_r, val_r, test_r = [], [], []
    elif HAS_SKMULTILEARN:
        print("\n── Running IterativeStratification (skmultilearn) ──────────────────")
        train_r, val_r, test_r = iterative_stratify(remaining, all_data)
    else:
        print("\n── Running greedy multi-label stratification (fallback) ─────────────")
        train_r, val_r, test_r = greedy_multilabel_stratify(remaining, all_data)

    # ── Step 5: Merge seed + stratified pools ────────────────────────────
    train_idx = train_seed + list(train_r)
    val_idx   = val_seed   + list(val_r)
    test_idx  = test_seed  + list(test_r)

    # ── Step 6: Validate ─────────────────────────────────────────────────
    valid = validate_splits(train_idx, val_idx, test_idx, all_data)
    if not valid:
        raise RuntimeError(
            "Split validation FAILED — fix class imbalance before proceeding."
        )

    # ── Step 7: Per-split distribution reports ───────────────────────────
    for name, idxs in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
        print_distribution([all_data[i] for i in idxs], f"{name} split")

    # ── Step 8: Compute & export class weights ───────────────────────────
    weights = compute_class_weights(train_idx, all_data)
    print("\n── Class weights for loss balancing (based on train box counts) ───")
    print(f"  {'Class':<15} {'Weight':>8}")
    for cls, w in sorted(weights.items(), key=lambda x: -x[1]):
        print(f"  {cls:<15} {w:>8.4f}")

    weights_path = NEW_ROOT / "class_weights.json"
    NEW_ROOT.mkdir(parents=True, exist_ok=True)
    with open(weights_path, "w") as f:
        json.dump({"class_weights": weights, "classes": FINAL_CLASSES}, f, indent=2)
    print(f"\n  Saved class weights → {weights_path}")

    # ── Step 9: Write split summary JSON ─────────────────────────────────
    summary = {
        "total":  len(all_data),
        "train":  len(train_idx),
        "val":    len(val_idx),
        "test":   len(test_idx),
        "ratios": {
            "train": round(len(train_idx) / len(all_data), 3),
            "val":   round(len(val_idx)   / len(all_data), 3),
            "test":  round(len(test_idx)  / len(all_data), 3),
        },
        "classes": FINAL_CLASSES,
        "stratification": "iterative" if HAS_SKMULTILEARN else "greedy_multilabel",
    }
    with open(NEW_ROOT / "split_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    # ── Step 10: Write files ──────────────────────────────────────────────
    print("\n── Writing split directories ───────────────────────────────────────")
    create_split_files(train_idx, all_data, "train")
    create_split_files(val_idx,   all_data, "val")
    create_split_files(test_idx,  all_data, "test")

    # ── Final summary ─────────────────────────────────────────────────────
    print(f"""
╔══════════════════════════════════════════════════════════════╗
║                      SPLIT COMPLETE ✅                       ║
╠══════════════════════════════════════════════════════════════╣
║  Total images : {len(all_data):<7}                                  ║
║  Train        : {len(train_idx):<7} ({len(train_idx)/len(all_data)*100:.1f}%)                           ║
║  Val          : {len(val_idx):<7} ({len(val_idx)/len(all_data)*100:.1f}%)                            ║
║  Test         : {len(test_idx):<7} ({len(test_idx)/len(all_data)*100:.1f}%)                            ║
║  Classes      : {N_CLASSES:<7}                                  ║
║  Strategy     : {'iterative' if HAS_SKMULTILEARN else 'greedy_multilabel':<20}                   ║
╚══════════════════════════════════════════════════════════════╝

📂 Output  : {NEW_ROOT}
⚖️  Weights : {weights_path}

Next step — use class weights in your training loss, e.g.:
  weights = json.load(open('class_weights.json'))['class_weights']
  # PyTorch:
  weight_tensor = torch.tensor([weights[c] for c in FINAL_CLASSES])
  criterion = FocalLoss(alpha=weight_tensor, gamma=2.0)
""")


if __name__ == "__main__":
    main()


── Scanning original dataset folders ──────────────────────────────
  train_set: 4368 XML files found
  validation_set: 1310 XML files found
  test_set: 2184 XML files found

✅  Collected 7862 valid pairs. Skipped 0.

──────────────────────────────────────────────────────────────────────
  Full merged dataset  (7862 images | 122295 bounding boxes)
──────────────────────────────────────────────────────────────────────
  Class             Images    Img%     Boxes    Box%
  ─────────────── ──────── ───────  ──────── ───────
  CYPRO               5335   67.9%     12102    9.9%
  CYPRO_max           5711   72.6%     16807   13.7%
  CYPRO_min           5395   68.6%     14185   11.6%
  ECHCG               1475   18.8%      5533    4.5%
  ECHCG_Ve            2433   30.9%      5628    4.6%
  NC                  4431   56.4%     11215    9.2%
  SOLNI_V1            1268   16.1%      8774    7.2%
  SOLNI_Vc            3993   50.8%     13902   11.4%
  ZEAMX_V1            5408   68.8%     23345   1

MODEL TRAINING LOGIC, CHANGE THE CELL BELOW

In [ ]:
"""
WeedMaize — YOLOv11 Training Pipeline
======================================
Fixes applied vs original:
  1. Paths aligned to stratified_dataset structure
  2. Class names sourced from LABEL_MAP (not re-scanned from XML)
     → prevents OE / POROL / unmapped labels leaking back in
  3. convert_box argument order fixed: was (xmin,xmax,ymin,ymax)
     → correct is (xmin,ymin,xmax,ymax) — wrong order = broken boxes
  4. Class weights wired into loss via class_weights.json
  5. Symlink-safe file copy (resolves symlink before copying)
  6. label_file properly closed via context manager
  7. Skips boxes whose label is not in class_map (safe guard)
  8. imgsz bumped to 1280 — drone images have tiny weeds, 640 loses detail
"""

# ── 0. Install & imports ────────────────────────────────────────────────────
# !pip install ultralytics   # uncomment if not installed

import os
import json
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

from ultralytics import YOLO

# ═══════════════════════════════════════════════════════════════════════════
# 1. PATHS  (all derived from single base_dir)
# ═══════════════════════════════════════════════════════════════════════════

BASE_DIR   = Path("/kaggle/working/stratified_dataset")
TRAIN_DIR  = BASE_DIR / "train"
VAL_DIR    = BASE_DIR / "val"
TEST_DIR   = BASE_DIR / "test"
YOLO_DIR   = BASE_DIR / "weed_yolo"          # YOLO-format output
WEIGHTS_F  = BASE_DIR / "class_weights.json"

SPLITS = {
    "train": TRAIN_DIR,
    "val":   VAL_DIR,
    "test":  TEST_DIR,
}

# ═══════════════════════════════════════════════════════════════════════════
# 2. CLASS NAMES  — must match stratification LABEL_MAP exactly
#    Do NOT re-scan XMLs: dropped classes (OE, POROL) would re-appear
# ═══════════════════════════════════════════════════════════════════════════

CLASSES = [
    "CYPRO",
    "CYPRO_max",
    "CYPRO_min",
    "ECHCG",
    "ECHCG_Ve",
    "NC",
    "SOLNI_V1",
    "SOLNI_Vc",
    "ZEAMX_V1",
    "ZEAMX_V3",
    "ZEAMX_V4",
]
CLASS_MAP = {name: i for i, name in enumerate(CLASSES)}
print(f"Classes ({len(CLASSES)}): {CLASSES}")

# ═══════════════════════════════════════════════════════════════════════════
# 3. CREATE YOLO FOLDER STRUCTURE
# ═══════════════════════════════════════════════════════════════════════════

for split in SPLITS:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("✅ YOLO directory structure created.")

# ═══════════════════════════════════════════════════════════════════════════
# 4. VOC → YOLO CONVERSION
#    FIXED argument order: (xmin, ymin, xmax, ymax)  ← was wrong before
# ═══════════════════════════════════════════════════════════════════════════

def convert_box(img_w: int, img_h: int,
                xmin: float, ymin: float,
                xmax: float, ymax: float) -> tuple:
    """Convert Pascal VOC absolute coords → YOLO normalised (cx,cy,w,h)."""
    cx = ((xmin + xmax) / 2) / img_w
    cy = ((ymin + ymax) / 2) / img_h
    bw = (xmax - xmin) / img_w
    bh = (ymax - ymin) / img_h
    return cx, cy, bw, bh


total_imgs, total_boxes, skipped_boxes = 0, 0, 0

for split, src_dir in SPLITS.items():
    img_dst = YOLO_DIR / "images" / split
    lbl_dst = YOLO_DIR / "labels" / split
    split_imgs = split_boxes = 0

    for file in sorted(os.listdir(src_dir)):
        if not file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        img_path = Path(src_dir) / file
        xml_path = Path(src_dir) / (Path(file).stem + ".xml")

        # ── Symlink-safe copy ──────────────────────────────────────────
        dst_img = img_dst / file
        shutil.copy2(img_path.resolve(), dst_img)   # resolves symlinks
        split_imgs += 1

        if not xml_path.exists():
            continue  # image without annotation — still copied

        tree    = ET.parse(xml_path)
        root_el = tree.getroot()
        size    = root_el.find("size")
        img_w   = int(size.find("width").text)
        img_h   = int(size.find("height").text)

        label_path = lbl_dst / (Path(file).stem + ".txt")
        with open(label_path, "w") as lf:
            for obj in root_el.findall("object"):
                name = obj.find("name").text

                # Skip any label not in our final class map (safety guard)
                if name not in CLASS_MAP:
                    skipped_boxes += 1
                    continue

                cls_id = CLASS_MAP[name]
                box    = obj.find("bndbox")
                xmin   = float(box.find("xmin").text)
                ymin   = float(box.find("ymin").text)
                xmax   = float(box.find("xmax").text)
                ymax   = float(box.find("ymax").text)

                cx, cy, bw, bh = convert_box(img_w, img_h, xmin, ymin, xmax, ymax)
                lf.write(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
                split_boxes += 1

    print(f"  {split:<6}: {split_imgs} images | {split_boxes} boxes")
    total_imgs  += split_imgs
    total_boxes += split_boxes

print(f"\n✅ Conversion done — {total_imgs} images | {total_boxes} boxes written")
if skipped_boxes:
    print(f"⚠️  {skipped_boxes} boxes skipped (labels not in CLASS_MAP)")

# ═══════════════════════════════════════════════════════════════════════════
# 5. DATASET YAML
# ═══════════════════════════════════════════════════════════════════════════

yaml_lines = [
    f"path: {YOLO_DIR}",
    "train: images/train",
    "val:   images/val",
    "test:  images/test",
    "",
    f"nc: {len(CLASSES)}",
    "",
    "names:",
]
for i, c in enumerate(CLASSES):
    yaml_lines.append(f"  {i}: {c}")

yaml_text = "\n".join(yaml_lines) + "\n"
yaml_path = YOLO_DIR / "dataset.yaml"
yaml_path.write_text(yaml_text)
print(f"\n✅ dataset.yaml written → {yaml_path}")
print(yaml_text)

# ═══════════════════════════════════════════════════════════════════════════
# 6. LOAD CLASS WEIGHTS  (from stratification step)
# ═══════════════════════════════════════════════════════════════════════════

if WEIGHTS_F.exists():
    with open(WEIGHTS_F) as f:
        cw_data = json.load(f)
    class_weights = [cw_data["class_weights"].get(c, 1.0) for c in CLASSES]
    print("\n⚖️  Class weights loaded:")
    for c, w in zip(CLASSES, class_weights):
        print(f"   {c:<15} {w:.4f}")
else:
    class_weights = None
    print("⚠️  class_weights.json not found — training without class weights.")

# ═══════════════════════════════════════════════════════════════════════════
# 7. TRAIN YOLOv11
# ═══════════════════════════════════════════════════════════════════════════
#
# T4 GPU VRAM budget (14.9 GB each):
#
#   imgsz  | batch/GPU | approx VRAM/GPU | status
#   -------|-----------|-----------------|-------
#   1280   |    16     |    ~15.5 GB     | ❌ OOM  ← what crashed
#    832   |     8     |    ~11.0 GB     | ✅ safe
#    640   |    16     |    ~8.5  GB     | ✅ safe (fallback)
#
# cache="ram" loaded 12.4 GB into RAM which competed with VRAM via
# DDP shared memory — switched to "disk" which is deterministic too.
# env var PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True reduces
# fragmentation on T4s running DDP.

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

model = YOLO("yolo11m.pt")


model.train(
    data        = str(yaml_path),
    epochs      = 100,

    # ── Hardware ──────────────────────────────────────────────────────────
    device      = [0, 1],      # both T4 GPUs
    batch       = 16,          # 8 per GPU — fits at 832px on T4 (11GB/GPU)
    workers     = 4,           # 4 total (2 per GPU); more causes RAM pressure
    cache       = "disk",      # avoids 12GB RAM competing with VRAM in DDP

    # ── Resolution ────────────────────────────────────────────────────────
    # 832 is best T4-safe resolution for small weed detection.
    # It's 30% more pixels than 640 — catches small plants 640 misses.
    # Fall back to imgsz=640, batch=16 only if you still OOM here.
    imgsz       = 832,

    # ── Augmentation ──────────────────────────────────────────────────────
    hsv_h       = 0.015,       # hue jitter — field lighting variation
    hsv_s       = 0.7,         # saturation jitter
    hsv_v       = 0.4,         # brightness jitter
    degrees     = 15,          # rotation — weeds grow at any angle
    translate   = 0.1,
    scale       = 0.6,         # scale jitter — simulates drone altitude change
    shear       = 2,
    perspective = 0.0005,      # mild warp for drone tilt
    fliplr      = 0.5,
    flipud      = 0.3,         # drone top-down = no natural "up"
    mosaic      = 1.0,
    mixup       = 0.15,
    copy_paste  = 0.2,         # pastes rare class instances into other images

    # ── Regularisation ────────────────────────────────────────────────────
    erasing     = 0.4,
    dropout     = 0.0,
    weight_decay= 0.0005,
    close_mosaic= 15,          # disable mosaic for final 15 epochs

    # ── Loss weights ──────────────────────────────────────────────────────
    cls         = 0.5,
    box         = 7.5,
    dfl         = 1.5,

    # ── Training behaviour ────────────────────────────────────────────────
    patience    = 25,
    optimizer   = "AdamW",
    lr0         = 0.001,
    lrf         = 0.01,
    warmup_epochs = 3,
    cos_lr      = True,
    amp         = True,

    # ── Logging & checkpoints ─────────────────────────────────────────────
    project     = "/kaggle/working/runs",
    name        = "weedmaize_yolo11m",
    exist_ok    = True,
    plots       = True,
    save        = True,
    save_period = 10,          # checkpoint every 10 epochs (Kaggle time limit safety)
    val         = True,
)

print("\n🎉 Training complete!")
print("   Best weights → /kaggle/working/runs/weedmaize_yolo11m/weights/best.pt")

# ── Save results to output dir ─────────────────────────────────────────────
import shutil
results_src = Path("/kaggle/working/runs/weedmaize_yolo11m")
results_dst = Path("/kaggle/working/yolo_results")
results_dst.mkdir(exist_ok=True)

shutil.copy(results_src / "weights" / "best.pt", results_dst / "best_model.pt")
shutil.copy(results_src / "weights" / "last.pt", results_dst / "last_model.pt")

for plot in results_src.glob("*.png"):
    shutil.copy(plot, results_dst / plot.name)
for plot in results_src.glob("*.csv"):
    shutil.copy(plot, results_dst / plot.name)

print(f"   Results saved → {results_dst}")
print(f"   Files: {sorted(os.listdir(results_dst))}")

Classes (11): ['CYPRO', 'CYPRO_max', 'CYPRO_min', 'ECHCG', 'ECHCG_Ve', 'NC', 'SOLNI_V1', 'SOLNI_Vc', 'ZEAMX_V1', 'ZEAMX_V3', 'ZEAMX_V4']
✅ YOLO directory structure created.
  train : 6284 images | 96737 boxes
  val   : 786 images | 12366 boxes
  test  : 792 images | 12796 boxes

✅ Conversion done — 7862 images | 121899 boxes written
⚠️  416 boxes skipped (labels not in CLASS_MAP)

✅ dataset.yaml written → /kaggle/working/stratified_dataset/weed_yolo/dataset.yaml
path: /kaggle/working/stratified_dataset/weed_yolo
train: images/train
val:   images/val
test:  images/test

nc: 11

names:
  0: CYPRO
  1: CYPRO_max
  2: CYPRO_min
  3: ECHCG
  4: ECHCG_Ve
  5: NC
  6: SOLNI_V1
  7: SOLNI_Vc
  8: ZEAMX_V1
  9: ZEAMX_V3
  10: ZEAMX_V4


⚖️  Class weights loaded:
   CYPRO           0.9174
   CYPRO_max       0.6615
   CYPRO_min       0.7828
   ECHCG           2.0093
   ECHCG_Ve        1.9952
   NC              0.9879
   SOLNI_V1        1.3505
   SOLNI_Vc        0.7895
   ZEAMX_V1        0.4722
  

MODEL EFFICIENCY BENCHMARK

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MODEL EFFICIENCY BENCHMARK (Params, GFLOPs, Latency, FPS, Size)
# ═══════════════════════════════════════════════════════════════════════════

import torch
import time
import pandas as pd
from thop import profile
from ultralytics import YOLO

MODEL_PATH = results_dst / "best_model.pt"

model = YOLO(str(MODEL_PATH))
net = model.model

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net = net.to(device).eval()

# -------------------------------------------------
# Params
# -------------------------------------------------
params = sum(p.numel() for p in net.parameters())
params_m = params / 1e6

# -------------------------------------------------
# GFLOPs
# -------------------------------------------------
dummy = torch.randn(1, 3, 832, 832).to(device)

macs, _ = profile(net, inputs=(dummy,), verbose=False)
gflops = macs / 1e9

# -------------------------------------------------
# Model size
# -------------------------------------------------
model_size_mb = MODEL_PATH.stat().st_size / (1024 ** 2)

# -------------------------------------------------
# Latency / FPS benchmark
# -------------------------------------------------
warmup = 20
runs = 100

with torch.no_grad():

    for _ in range(warmup):
        _ = net(dummy)

    torch.cuda.synchronize()
    start = time.time()

    for _ in range(runs):
        _ = net(dummy)

    torch.cuda.synchronize()
    end = time.time()

latency = (end - start) / runs
fps = 1 / latency

# -------------------------------------------------
# Print results
# -------------------------------------------------
print("\n📊 MODEL EFFICIENCY")
print(f"Params (M): {params_m:.2f}")
print(f"GFLOPs: {gflops:.2f}")
print(f"Model Size (MB): {model_size_mb:.2f}")
print(f"Latency (ms): {latency*1000:.2f}")
print(f"FPS: {fps:.2f}")

# -------------------------------------------------
# Save for paper
# -------------------------------------------------
efficiency = pd.DataFrame([{
    "model": "YOLOv11m",
    "params_M": round(params_m,3),
    "GFLOPs": round(gflops,3),
    "size_MB": round(model_size_mb,3),
    "latency_ms": round(latency*1000,3),
    "FPS": round(fps,3)
}])

efficiency_path = results_dst / "model_efficiency.csv"
efficiency.to_csv(efficiency_path, index=False)

print(f"\n📄 Efficiency table saved → {efficiency_path}")

MODEL OUTPUT

In [ ]:
import shutil
import os
from pathlib import Path

# Base YOLO runs directory
runs_dir = Path("/kaggle/working/runs/detect")

# Find the newest training run
train_dirs = sorted(runs_dir.glob("train*"), key=os.path.getmtime)
latest_run = train_dirs[-1]

print("Latest run found:", latest_run)

# Folder to store results
save_dir = Path("/kaggle/working/yolo_results")
save_dir.mkdir(exist_ok=True)

# Copy entire experiment folder
shutil.copytree(latest_run, save_dir / "experiment", dirs_exist_ok=True)

# Copy model weights
weights_dir = latest_run / "weights"

shutil.copy(weights_dir / "best.pt", save_dir / "best_model.pt")
shutil.copy(weights_dir / "last.pt", save_dir / "last_model.pt")

print("Saved files:")
print(os.listdir(save_dir))

In [ ]:
# ============================================================
#  TEACHER MODEL EVALUATION  — YOLOv11 Weed Detection
#  Run this cell AFTER training is complete.
#  Produces all plots, CSVs, and a KD-ready summary JSON.
# ============================================================

import os, shutil, json, zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import cv2
from ultralytics import YOLO

# ──────────────────────────────────────────────────────────────
#  CONFIG  —  edit these two lines if your paths differ
# ──────────────────────────────────────────────────────────────
BASE_DIR   = "/kaggle/working/stratified_dataset"
YOLO_DIR   = f"{BASE_DIR}/weed_dataset"          # must match training script
YAML_PATH  = f"{YOLO_DIR}/dataset.yaml"
RUNS_DIR   = Path("/kaggle/working/runs/detect")
EXPORT_ROOT = Path("/kaggle/working/teacher_export")

# ──────────────────────────────────────────────────────────────
#  0.  SANITY-CHECK YAML EXISTS
# ──────────────────────────────────────────────────────────────
if not os.path.exists(YAML_PATH):
    raise FileNotFoundError(
        f"dataset.yaml not found at {YAML_PATH}\n"
        "Make sure YOLO_DIR matches the path used during training."
    )
print(f"[OK] dataset.yaml found: {YAML_PATH}")

# ──────────────────────────────────────────────────────────────
#  1.  LOCATE LATEST TRAINING RUN
# ──────────────────────────────────────────────────────────────
train_dirs = sorted(RUNS_DIR.glob("train*"), key=os.path.getmtime)
if not train_dirs:
    raise RuntimeError(f"No training runs found in {RUNS_DIR}")
latest_run = train_dirs[-1]
print(f"[INFO] Using run: {latest_run}")

# Output folders
plots_dir = EXPORT_ROOT / "plots"
csv_dir   = EXPORT_ROOT / "csv"
model_dir = EXPORT_ROOT / "models"
pred_dir  = EXPORT_ROOT / "predictions"
for d in [plots_dir, csv_dir, model_dir, pred_dir]:
    d.mkdir(parents=True, exist_ok=True)

# ──────────────────────────────────────────────────────────────
#  2.  SAVE WEIGHTS
# ──────────────────────────────────────────────────────────────
weights_src = latest_run / "weights"
shutil.copy(weights_src / "best.pt", model_dir / "best.pt")
shutil.copy(weights_src / "last.pt", model_dir / "last.pt")
print("[INFO] Weights saved.")

# ──────────────────────────────────────────────────────────────
#  3.  LOAD TRAINING METRICS CSV
# ──────────────────────────────────────────────────────────────
df = pd.read_csv(latest_run / "results.csv")
df.columns = df.columns.str.strip()
df.to_csv(csv_dir / "training_metrics.csv", index=False)

epochs = df["epoch"] if "epoch" in df.columns else pd.Series(np.arange(len(df)))
print(f"[INFO] Training metrics loaded ({len(df)} epochs).")

# ── helper ─────────────────────────────────────────────────────
BLUE, RED, GREEN, PURPLE = "#2563EB", "#DC2626", "#16A34A", "#9333EA"

def scol(df, *names):
    """Case-insensitive column lookup."""
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    return None

# ──────────────────────────────────────────────────────────────
#  4.  TRAINING CURVE PLOTS
# ──────────────────────────────────────────────────────────────

# --- Loss curves ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Training & Validation Loss Curves", fontsize=14, fontweight="bold")
for ax, key, title in zip(axes,
                           ["box_loss", "cls_loss", "dfl_loss"],
                           ["Box Loss", "Class Loss", "DFL Loss"]):
    tc = scol(df, f"train/{key}", f"train_{key}")
    vc = scol(df, f"val/{key}",   f"val_{key}")
    if tc: ax.plot(epochs, df[tc], color=BLUE, label="Train", linewidth=1.8)
    if vc: ax.plot(epochs, df[vc], color=RED,  label="Val",   linewidth=1.8)
    ax.set_title(title, fontweight="bold"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(plots_dir / "01_loss_curves.png", dpi=200, bbox_inches="tight")
plt.close(); print("[PLOT] 01_loss_curves.png")

# --- mAP curves ---
map50_col = scol(df, "metrics/mAP50(B)",    "metrics/map50",    "map50")
map95_col = scol(df, "metrics/mAP50-95(B)", "metrics/map50-95", "mAP50-95(B)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Mean Average Precision (mAP)", fontsize=14, fontweight="bold")
for ax, col, label in zip(axes, [map50_col, map95_col], ["mAP@0.50", "mAP@0.50:0.95"]):
    if col:
        ax.plot(epochs, df[col], color=GREEN, linewidth=2)
        ax.fill_between(epochs, df[col], alpha=0.15, color=GREEN)
        best_i = df[col].idxmax()
        ax.annotate(f"Best: {df[col].max():.4f}\n@ ep {int(epochs.iloc[best_i])}",
                    xy=(epochs.iloc[best_i], df[col].max()),
                    xytext=(10, -20), textcoords="offset points",
                    arrowprops=dict(arrowstyle="->"), fontsize=8)
    ax.set_title(label, fontweight="bold"); ax.set_xlabel("Epoch"); ax.set_ylabel("mAP")
    ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(plots_dir / "02_map_curves.png", dpi=200, bbox_inches="tight")
plt.close(); print("[PLOT] 02_map_curves.png")

# --- Precision & Recall ---
fig, ax = plt.subplots(figsize=(8, 4))
pc = scol(df, "metrics/precision(B)", "metrics/precision", "precision")
rc = scol(df, "metrics/recall(B)",    "metrics/recall",    "recall")
if pc: ax.plot(epochs, df[pc], color=BLUE, label="Precision", linewidth=1.8)
if rc: ax.plot(epochs, df[rc], color=RED,  label="Recall",    linewidth=1.8)
ax.set_title("Precision & Recall over Training", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
ax.legend(); ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(plots_dir / "03_precision_recall_curves.png", dpi=200, bbox_inches="tight")
plt.close(); print("[PLOT] 03_precision_recall_curves.png")

# --- LR schedule ---
lr_cols = [c for c in df.columns if "lr" in c.lower()]
if lr_cols:
    fig, ax = plt.subplots(figsize=(8, 3))
    for col in lr_cols:
        ax.plot(epochs, df[col], label=col, linewidth=1.5)
    ax.set_title("Learning Rate Schedule", fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("LR")
    ax.legend(fontsize=8); ax.grid(linestyle="--", alpha=0.4)
    plt.tight_layout()
    fig.savefig(plots_dir / "04_lr_schedule.png", dpi=200, bbox_inches="tight")
    plt.close(); print("[PLOT] 04_lr_schedule.png")

# --- Summary dashboard ---
fig = plt.figure(figsize=(18, 10))
fig.suptitle("YOLOv11 Teacher — Training Summary", fontsize=16, fontweight="bold")
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

panel_cfg = [
    (gs[0,0], "train/box_loss","train_box_loss", "val/box_loss","val_box_loss", "Box Loss"),
    (gs[0,1], "train/cls_loss","train_cls_loss", "val/cls_loss","val_cls_loss", "Cls Loss"),
    (gs[0,2], "train/dfl_loss","train_dfl_loss", "val/dfl_loss","val_dfl_loss", "DFL Loss"),
]
for spec, tk1, tk2, vk1, vk2, title in panel_cfg:
    ax = fig.add_subplot(spec)
    tc = scol(df, tk1, tk2); vc = scol(df, vk1, vk2)
    if tc: ax.plot(epochs, df[tc], color=BLUE, label="Train")
    if vc: ax.plot(epochs, df[vc], color=RED,  label="Val")
    ax.set_title(title); ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[1,0])
if map50_col:
    ax4.plot(epochs, df[map50_col], color=GREEN, linewidth=2)
    ax4.fill_between(epochs, df[map50_col], alpha=0.15, color=GREEN)
ax4.set_title("mAP@0.50"); ax4.grid(alpha=0.3)

ax5 = fig.add_subplot(gs[1,1])
if map95_col:
    ax5.plot(epochs, df[map95_col], color=PURPLE, linewidth=2)
    ax5.fill_between(epochs, df[map95_col], alpha=0.15, color=PURPLE)
ax5.set_title("mAP@0.50:0.95"); ax5.grid(alpha=0.3)

ax6 = fig.add_subplot(gs[1,2])
if pc: ax6.plot(epochs, df[pc], color=BLUE, label="Precision")
if rc: ax6.plot(epochs, df[rc], color=RED,  label="Recall")
ax6.set_title("Precision & Recall"); ax6.legend(fontsize=7); ax6.grid(alpha=0.3)

fig.savefig(plots_dir / "05_summary_dashboard.png", dpi=200, bbox_inches="tight")
plt.close(); print("[PLOT] 05_summary_dashboard.png")

# Copy YOLO auto-generated plots
for p in list(latest_run.glob("*.png")) + list(latest_run.glob("*.jpg")):
    shutil.copy(p, plots_dir / p.name)
    print(f"[COPY] {p.name}")

# ──────────────────────────────────────────────────────────────
#  5.  LOAD TEACHER MODEL
# ──────────────────────────────────────────────────────────────
teacher = YOLO(str(model_dir / "best.pt"))
names   = teacher.names   # {0: 'class_name', ...}
n_cls   = len(names)

# ──────────────────────────────────────────────────────────────
#  6.  STANDARD TEST-SET VALIDATION
# ──────────────────────────────────────────────────────────────
test_results = teacher.val(
    data=YAML_PATH,
    split="test",
    imgsz=640,
    batch=16,
    verbose=True,
    save_json=True,
    project=str(EXPORT_ROOT),
    name="test_eval"
)

box = test_results.box

# Per-class metrics
rows = []
for i, name in names.items():
    p  = float(box.p[i])  if hasattr(box, "p")    else 0.0
    r  = float(box.r[i])  if hasattr(box, "r")    else 0.0
    a50= float(box.ap50[i]) if hasattr(box,"ap50") else 0.0
    a95= float(box.ap[i])  if hasattr(box, "ap")  else 0.0
    f1 = 2*p*r/(p+r+1e-9)
    rows.append(dict(class_id=i, class_name=name,
                     precision=p, recall=r, f1=f1,
                     mAP50=a50, mAP50_95=a95))

per_class_df = pd.DataFrame(rows)
per_class_df.to_csv(csv_dir / "test_per_class_metrics.csv", index=False)

overall = dict(
    mAP50    = float(box.map50),
    mAP50_95 = float(box.map),
    precision= float(box.mp),
    recall   = float(box.mr),
    f1       = float(2*box.mp*box.mr/(box.mp+box.mr+1e-9)),
    timestamp= datetime.now().isoformat(),
    model    = "yolo11m-best.pt",
    dataset  = "Zenodo-5106795",
)
pd.DataFrame([overall]).to_csv(csv_dir / "test_overall_summary.csv", index=False)
print(f"[CSV] Overall → mAP50={overall['mAP50']:.4f}  mAP50-95={overall['mAP50_95']:.4f}")

# ──────────────────────────────────────────────────────────────
#  7.  PLOT  Per-class bar chart  (P / R / mAP50 / mAP95 / F1)
# ──────────────────────────────────────────────────────────────
metrics_bar = [
    ("precision", "Precision",   BLUE),
    ("recall",    "Recall",      RED),
    ("mAP50",     "mAP@0.50",    GREEN),
    ("mAP50_95",  "mAP@0.50:95", PURPLE),
    ("f1",        "F1 Score",    "#EA580C"),
]
fig, axes = plt.subplots(1, len(metrics_bar), figsize=(5*len(metrics_bar), 5))
fig.suptitle("Per-Class Metrics on Test Set (Teacher)", fontsize=14, fontweight="bold")
x = np.arange(n_cls)

for ax, (col, label, color) in zip(axes, metrics_bar):
    vals = per_class_df[col].astype(float).values
    bars = ax.bar(x, vals, color=color, alpha=0.82, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(per_class_df["class_name"], rotation=35, ha="right", fontsize=7)
    ax.set_title(label, fontweight="bold"); ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=6)

plt.tight_layout()
fig.savefig(plots_dir / "06_per_class_metrics.png", dpi=200, bbox_inches="tight")
plt.close(); print("[PLOT] 06_per_class_metrics.png")

# ──────────────────────────────────────────────────────────────
#  8.  SPEED METRICS
# ──────────────────────────────────────────────────────────────
speed = test_results.speed   # {'preprocess': ms, 'inference': ms, 'postprocess': ms}
speed_df = pd.DataFrame([speed])
speed_df.to_csv(csv_dir / "speed_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4))
labels_s = list(speed.keys())
vals_s   = [speed[k] for k in labels_s]
bars = ax.bar(labels_s, vals_s, color=[BLUE, GREEN, RED], alpha=0.85, edgecolor="white")
for bar, val in zip(bars, vals_s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f"{val:.2f} ms", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Inference Speed per Image (ms)", fontweight="bold")
ax.set_ylabel("Milliseconds"); ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(plots_dir / "07_speed_metrics.png", dpi=200, bbox_inches="tight")
plt.close(); print("[PLOT] 07_speed_metrics.png")

# ──────────────────────────────────────────────────────────────
#  9.  CONFIDENCE DISTRIBUTION  (KD teacher calibration)
# ──────────────────────────────────────────────────────────────
test_img_dir = Path(YOLO_DIR) / "images" / "test"
test_images  = sorted(test_img_dir.glob("*.jpg"))[:200] + \
               sorted(test_img_dir.glob("*.png"))[:200]
test_images  = test_images[:200]

all_confs   = []
all_classes = []
box_areas   = []

if test_images:
    preds = teacher.predict(
        source=[str(p) for p in test_images],
        imgsz=640, conf=0.01,   # low threshold → capture full distribution
        iou=0.5, verbose=False
    )
    for result in preds:
        if result.boxes is not None and len(result.boxes):
            confs = result.boxes.conf.cpu().numpy()
            clss  = result.boxes.cls.cpu().numpy().astype(int)
            xyxy  = result.boxes.xyxy.cpu().numpy()
            areas = (xyxy[:,2]-xyxy[:,0]) * (xyxy[:,3]-xyxy[:,1])
            all_confs.extend(confs.tolist())
            all_classes.extend(clss.tolist())
            box_areas.extend(areas.tolist())

all_confs   = np.array(all_confs)
all_classes = np.array(all_classes)
box_areas   = np.array(box_areas)

# Confidence histogram (overall + per class)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Teacher Confidence Distribution (KD Calibration)", fontweight="bold", fontsize=13)

axes[0].hist(all_confs, bins=40, color=BLUE, alpha=0.8, edgecolor="white")
axes[0].axvline(0.25, color=RED,    linestyle="--", label="conf=0.25")
axes[0].axvline(0.50, color=GREEN,  linestyle="--", label="conf=0.50")
axes[0].set_title("Overall Confidence Distribution")
axes[0].set_xlabel("Confidence Score"); axes[0].set_ylabel("Count")
axes[0].legend(); axes[0].grid(linestyle="--", alpha=0.4)

colors_cls = plt.cm.tab20(np.linspace(0, 1, n_cls))
for i, name in names.items():
    mask = all_classes == i
    if mask.sum() > 0:
        axes[1].hist(all_confs[mask], bins=30, alpha=0.55,
                     label=name, edgecolor="white", color=colors_cls[i % len(colors_cls)])
axes[1].set_title("Per-Class Confidence Distribution")
axes[1].set_xlabel("Confidence Score"); axes[1].set_ylabel("Count")
axes[1].legend(fontsize=7, ncol=2); axes[1].grid(linestyle="--", alpha=0.4)

plt.tight_layout()
fig.savefig(plots_dir / "08_confidence_distribution.png", dpi=200, bbox_inches="tight")
plt.close(); print("[PLOT] 08_confidence_distribution.png")

# ──────────────────────────────────────────────────────────────
#  10. mAP vs IoU THRESHOLD SWEEP  (NMS sensitivity)
# ──────────────────────────────────────────────────────────────
iou_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.75]
iou_map50_list = []
iou_map95_list = []

for iou_t in iou_thresholds:
    r = teacher.val(data=YAML_PATH, split="test",
                    imgsz=640, batch=16, iou=iou_t, verbose=False)
    iou_map50_list.append(float(r.box.map50))
    iou_map95_list.append(float(r.box.map))
    print(f"  IoU={iou_t:.2f}  mAP50={iou_map50_list[-1]:.4f}  mAP95={iou_map95_list[-1]:.4f}")

iou_df = pd.DataFrame(dict(iou_threshold=iou_thresholds,
                            mAP50=iou_map50_list,
                            mAP50_95=iou_map95_list))
iou_df.to_csv(csv_dir / "iou_sensitivity.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(iou_thresholds, iou_map50_list,  "o-", color=GREEN,  label="mAP@0.50",    linewidth=2)
ax.plot(iou_thresholds, iou_map95_list,  "s-", color=PURPLE, label="mAP@0.50:95", linewidth=2)
ax.set_title("mAP vs IoU Threshold (NMS Sensitivity)", fontweight="bold")
ax.set_xlabel("IoU Threshold"); ax.set_ylabel("mAP")
ax.legend(); ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(plots_dir / "09_iou_sensitivity.png", dpi=200, bbox_inches="tight")
plt.close(); print("[PLOT] 09_iou_sensitivity.png")

# ──────────────────────────────────────────────────────────────
#  11. CONFIDENCE THRESHOLD SWEEP  (P-R tradeoff)
# ──────────────────────────────────────────────────────────────
conf_thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6]
conf_sweep = []

for conf_t in conf_thresholds:
    r = teacher.val(data=YAML_PATH, split="test",
                    imgsz=640, batch=16, conf=conf_t, verbose=False)
    conf_sweep.append(dict(
        conf_threshold = conf_t,
        precision      = float(r.box.mp),
        recall         = float(r.box.mr),
        mAP50          = float(r.box.map50),
        f1             = float(2*r.box.mp*r.box.mr/(r.box.mp+r.box.mr+1e-9)),
    ))
    print(f"  conf={conf_t:.2f}  P={conf_sweep[-1]['precision']:.3f}  "
          f"R={conf_sweep[-1]['recall']:.3f}  mAP50={conf_sweep[-1]['mAP50']:.4f}")

conf_df = pd.DataFrame(conf_sweep)
conf_df.to_csv(csv_dir / "conf_threshold_sweep.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Confidence Threshold Sweep", fontweight="bold", fontsize=13)

for col, label, color in [("precision","Precision",BLUE),
                           ("recall","Recall",RED),
                           ("f1","F1",GREEN)]:
    axes[0].plot(conf_df["conf_threshold"], conf_df[col],
                 "o-", label=label, color=color, linewidth=2)
axes[0].set_title("P / R / F1 vs Confidence"); axes[0].set_xlabel("Confidence Threshold")
axes[0].set_ylabel("Score"); axes[0].legend(); axes[0].grid(linestyle="--", alpha=0.4)

axes[1].plot(conf_df["recall"], conf_df["precision"], "o-", color=PURPLE, linewidth=2)
for _, row in conf_df.iterrows():
    axes[1].annotate(f"{row['conf_threshold']:.2f}",
                     (row["recall"], row["precision"]),
                     textcoords="offset points", xytext=(4,4), fontsize=7)
axes[1].set_title("Precision–Recall Curve (conf sweep)")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1.05); axes[1].grid(linestyle="--", alpha=0.4)

plt.tight_layout()
fig.savefig(plots_dir / "10_conf_threshold_sweep.png", dpi=200, bbox_inches="tight")
plt.close(); print("[PLOT] 10_conf_threshold_sweep.png")

# ──────────────────────────────────────────────────────────────
#  12. BOX SIZE vs CONFIDENCE SCATTER  (small object difficulty)
# ──────────────────────────────────────────────────────────────
if len(box_areas) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    scatter = ax.scatter(box_areas, all_confs, c=all_classes,
                         cmap="tab20", alpha=0.35, s=8)
    ax.set_title("Bounding-Box Area vs Confidence (Teacher)", fontweight="bold")
    ax.set_xlabel("Box Area (pixels²)"); ax.set_ylabel("Confidence")
    ax.axhline(0.25, color=RED, linestyle="--", label="conf=0.25"); ax.legend()
    ax.grid(linestyle="--", alpha=0.3)
    plt.tight_layout()
    fig.savefig(plots_dir / "11_boxsize_vs_confidence.png", dpi=200, bbox_inches="tight")
    plt.close(); print("[PLOT] 11_boxsize_vs_confidence.png")

# ──────────────────────────────────────────────────────────────
#  13. SAMPLE PREDICTION GRID  (up to 12 images)
# ──────────────────────────────────────────────────────────────
sample_imgs = (sorted(test_img_dir.glob("*.jpg")) +
               sorted(test_img_dir.glob("*.png")))[:12]

if sample_imgs:
    preds_vis = teacher.predict(
        source=[str(p) for p in sample_imgs],
        imgsz=640, conf=0.25, iou=0.5, verbose=False
    )
    cols = 4
    rows_n = (len(preds_vis) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols*4, rows_n*3.5))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, result in zip(axes, preds_vis):
        img_rgb = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb); ax.set_title(Path(result.path).name, fontsize=7); ax.axis("off")
    fig.suptitle("Sample Test Predictions — Teacher (conf ≥ 0.25)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    fig.savefig(plots_dir / "12_sample_predictions.png", dpi=150, bbox_inches="tight")
    plt.close()
    for result in preds_vis:
        result.save(filename=str(pred_dir / (Path(result.path).stem + "_pred.jpg")))
    print(f"[PLOT] 12_sample_predictions.png  ({len(preds_vis)} images)")

# ──────────────────────────────────────────────────────────────
#  14. BEST EPOCH SUMMARY
# ──────────────────────────────────────────────────────────────
if map50_col:
    best_i = df[map50_col].idxmax()
    pd.DataFrame([df.iloc[best_i].to_dict() | {"note":"best_epoch"}])\
      .to_csv(csv_dir / "best_epoch_metrics.csv", index=False)
    print("[CSV] best_epoch_metrics.csv saved.")

# ──────────────────────────────────────────────────────────────
#  15. KD TEACHER SUMMARY JSON
#      Feed this directly into your student training script
# ──────────────────────────────────────────────────────────────
kd_summary = {
    "teacher_model"       : "yolo11m-best.pt",
    "dataset"             : "Zenodo-5106795",
    "timestamp"           : datetime.now().isoformat(),
    "overall_metrics"     : overall,
    "per_class_metrics"   : per_class_df.to_dict(orient="records"),
    "speed_ms"            : speed,
    "iou_sensitivity"     : iou_df.to_dict(orient="records"),
    "conf_sweep"          : conf_df.to_dict(orient="records"),
    "recommended_kd_conf" : float(conf_df.loc[conf_df["f1"].idxmax(), "conf_threshold"]),
    "class_names"         : names,
    "conf_distribution"   : {
        "mean"   : float(np.mean(all_confs))  if len(all_confs) else None,
        "median" : float(np.median(all_confs))if len(all_confs) else None,
        "std"    : float(np.std(all_confs))   if len(all_confs) else None,
        "p25"    : float(np.percentile(all_confs, 25)) if len(all_confs) else None,
        "p75"    : float(np.percentile(all_confs, 75)) if len(all_confs) else None,
    },
}
(EXPORT_ROOT / "teacher_kd_summary.json").write_text(
    json.dumps(kd_summary, indent=2, default=str))
print("[JSON] teacher_kd_summary.json saved.")

# ──────────────────────────────────────────────────────────────
#  16. COPY FULL EXPERIMENT + README + ZIP
# ──────────────────────────────────────────────────────────────
shutil.copytree(latest_run, EXPORT_ROOT / "experiment_full", dirs_exist_ok=True)

readme = f"""# YOLOv11 Teacher Model — Evaluation Export
Generated : {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Key Results (Test Set)
| Metric        | Value   |
|---------------|---------|
| mAP@0.50      | {overall['mAP50']:.4f} |
| mAP@0.50:0.95 | {overall['mAP50_95']:.4f} |
| Precision     | {overall['precision']:.4f} |
| Recall        | {overall['recall']:.4f} |
| F1            | {overall['f1']:.4f} |

## Plots
| File | Description |
|------|-------------|
| 01_loss_curves.png | Train/Val box, cls, dfl loss |
| 02_map_curves.png | mAP50 and mAP50-95 over epochs |
| 03_precision_recall_curves.png | P & R over epochs |
| 04_lr_schedule.png | Learning rate schedule |
| 05_summary_dashboard.png | 6-panel overview |
| 06_per_class_metrics.png | P/R/F1/mAP per class |
| 07_speed_metrics.png | Inference latency breakdown |
| 08_confidence_distribution.png | KD calibration histogram |
| 09_iou_sensitivity.png | mAP vs IoU threshold sweep |
| 10_conf_threshold_sweep.png | P-R-F1 vs confidence threshold |
| 11_boxsize_vs_confidence.png | Small-object detection analysis |
| 12_sample_predictions.png | Visual prediction grid |

## KD Usage
Load `teacher_kd_summary.json` in your student training script.
`recommended_kd_conf` = conf threshold that maximises teacher F1
(use as soft-label threshold for distillation).

## Model
YOLOv11-m | 100 epochs | imgsz=640 | multi-GPU | AMP
"""
(EXPORT_ROOT / "README.md").write_text(readme)

zip_path = "/kaggle/working/teacher_export_FINAL"
shutil.make_archive(zip_path, "zip", str(EXPORT_ROOT))
print(f"\n{'='*60}")
print(f"  ✅  ZIP ready  →  {zip_path}.zip")
print(f"{'='*60}\n")

try:
    from IPython.display import FileLink, display
    display(FileLink("teacher_export_FINAL.zip"))
except Exception:
    print("Download: kaggle kernels output <your-kernel> -p .")

In [ ]:
# ============================================================
# AUTO UPLOAD ZIP TO GOOGLE DRIVE
# Run this AFTER ZIP is created
# ============================================================

import os

upload_file = "/kaggle/working/teacher_export_FINAL.zip"

file_name = os.path.basename(upload_file)

gfile = drive.CreateFile({'title': file_name})
gfile.SetContentFile(upload_file)
gfile.Upload()

print("✅ Upload complete.")
print("Drive file ID:", gfile['id'])

In [ ]:
import os
os._exit(0)